# Physics-Informed Room Impulse Response Generation Framework

> **End-to-end** differentiable pipeline: LSTM EDC prediction → bifurcated FDN
> (early reflections + late reverb) → sign-sticky phase reconstruction →
> physics-informed & multi-resolution STFT training.

| Component | Key Detail |
|---|---|
| EDC Predictor | 2-layer LSTM, 256 hidden, 6 octave bands × 256 time-steps |
| Early Reflections | 43-tap delayed-sum convolution |
| Late Reverb | 16-loop DifferentiableFDN, log-space delay constraint 0–50 ms |
| Phase Reconstruction | Sign-sticky algorithm, stickiness = 0.90, reverse-differentiation |
| Physics Loss | Acoustic Continuity + Linearized Momentum via `torch.autograd.grad` (SIREN) |
| MR-STFT Loss | Windows [512, 1024, 2048], rectangular phase (cos/sin on unit circle) |
| Training | PyTorch AMP (`torch.amp.autocast` / `GradScaler`) |

In [ ]:
# ============================================================
# Cell 1 — Install Dependencies (Colab / fresh env)
# ============================================================
# Uncomment the line below when running on Google Colab:
# !pip install -q datasets huggingface_hub matplotlib pandas torch torchaudio

In [ ]:
# ============================================================
# Cell 2 — Imports & Constants
# ============================================================
from __future__ import annotations

import ast
import io
import json
import os
import random
import subprocess
import time
import wave
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader, Dataset

from datasets import Audio, load_dataset
from huggingface_hub import hf_hub_download

# ── Named constants (no magic numbers) ───────────────────────
INPUT_DIM = 24                  # room_dims(3) + src(3) + mic(3) + abs(1) + bands(6) + modal(8)
MODAL_FEAT_DIM = 8
METRICS_DIM = 10
OCTAVE_BANDS = ["125", "250", "500", "1000", "2000", "4000"]
DEFAULT_MAX_RIR_LEN = 32_000
SAMPLE_RATE = 16_000

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed: int, deterministic: bool = True) -> None:
    """Seed Python, NumPy, and PyTorch RNG state."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        if hasattr(torch.backends, "cudnn"):
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass

print(f"Device: {DEVICE}")

In [ ]:
# ============================================================
# Cell 3 — Helper Functions
# ============================================================

def _parse_json_field(val: Any) -> Any:
    """Safely parse a JSON/Python-literal string field from metadata."""
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return None
    return val


def compute_edc(rir: np.ndarray) -> np.ndarray:
    """Energy Decay Curve via Schroeder backward integration (dB)."""
    energy = np.cumsum(rir[::-1] ** 2, dtype=np.float64)[::-1]
    energy = energy / (float(np.max(energy)) + 1e-12)
    return 10.0 * np.log10(energy + 1e-12).astype(np.float32)


def downsample_edc_tensor(edc: np.ndarray, num_time_steps: int = 256) -> np.ndarray:
    idx = np.linspace(0, len(edc) - 1, num_time_steps).astype(np.int64)
    return edc[idx]


def compute_multiband_edc(
    rir: np.ndarray, sr: int = 16_000, num_time_steps: int = 256
) -> np.ndarray:
    """Lightweight: duplicate broadband EDC into six octave bands."""
    edc = compute_edc(rir)
    edc_ds = downsample_edc_tensor(edc, num_time_steps=num_time_steps)
    return np.repeat(edc_ds[:, None], len(OCTAVE_BANDS), axis=1).astype(np.float32)


def _safe_spacing(xs: List[float]) -> Tuple[float, float]:
    if len(xs) < 2:
        return 0.0, 0.0
    diffs = np.diff(np.array(sorted(xs), dtype=np.float32))
    return float(np.mean(diffs)), float(np.std(diffs))


def compute_room_modes(L: float, W: float, H: float) -> np.ndarray:
    """8-dimensional modal feature vector for a shoebox room."""
    if min(L, W, H) <= 0:
        raise ValueError("Room dimensions must be positive")
    c = 343.0
    axial = [c / 2.0 * n / dim for dim in (L, W, H) for n in range(1, 6)]
    axial = sorted(axial)
    n_below_300 = float(sum(1 for f in axial if f < 300.0))
    f_schroeder = float(
        2000.0 * np.sqrt(0.161 * (L * W * H) / max(L * W + W * H + L * H, 1e-9))
    )
    f_first_axial = float(axial[0]) if axial else 0.0
    mean_spacing, std_spacing = _safe_spacing(axial)
    modal_overlap = float(f_schroeder / max(f_first_axial, 1e-6))
    tang_ax_ratio = 1.0
    log_volume = float(np.log10(max(L * W * H, 1e-9)))
    return np.array(
        [n_below_300, f_schroeder, f_first_axial, mean_spacing,
         std_spacing, modal_overlap, tang_ax_ratio, log_volume],
        dtype=np.float32,
    )


def _extract_sample_id(path: str) -> Optional[str]:
    if not path:
        return None
    norm = path.replace("\\", "/")
    base = os.path.basename(norm)
    sid, _ = os.path.splitext(base)
    return sid.strip() or None


def _decode_audio(audio: Dict[str, Any]) -> np.ndarray:
    """Decode audio from HuggingFace dict to float32 numpy array."""
    if isinstance(audio, dict) and "array" in audio:
        return np.asarray(audio["array"], dtype=np.float32).ravel()
    if isinstance(audio, dict) and audio.get("path") and os.path.exists(audio["path"]):
        with wave.open(audio["path"], "rb") as wf:
            frames = wf.readframes(wf.getnframes())
            data = np.frombuffer(frames, dtype=np.int16)
    elif isinstance(audio, dict) and audio.get("bytes"):
        with wave.open(io.BytesIO(audio["bytes"]), "rb") as wf:
            frames = wf.readframes(wf.getnframes())
            data = np.frombuffer(frames, dtype=np.int16)
    else:
        raise ValueError("Audio sample cannot be decoded")
    data = np.asarray(data)
    if data.dtype == np.int16:
        return (data.astype(np.float32) / 32768.0).ravel()
    if data.dtype == np.int32:
        return (data.astype(np.float32) / 2147483648.0).ravel()
    return data.astype(np.float32).ravel()


def _pad_or_truncate(arr: np.ndarray, length: int) -> np.ndarray:
    if len(arr) >= length:
        return arr[:length]
    return np.pad(arr, (0, length - len(arr)))

In [ ]:
# ============================================================
# Cell 4 — RIRMegaDataset
# ============================================================

class RIRMegaDataset(Dataset):
    """Full HuggingFace-backed dataset for the RIR-Mega collection."""

    _HF_SPLIT_MAP = {"train": "train", "validation": "val", "test": "test", "val": "val"}

    def __init__(
        self,
        split: str = "train",
        max_rir_len: int = DEFAULT_MAX_RIR_LEN,
        num_time_steps: int = 256,
        sample_rate: int = 16_000,
        cache_dir: Optional[str] = None,
    ) -> None:
        super().__init__()
        self.split = split
        self.max_rir_len = max_rir_len
        self.num_time_steps = num_time_steps
        self.sample_rate = sample_rate

        hf_split = "validation" if split == "val" else split
        self._hf_ds = load_dataset(
            "mandipgoswami/rirmega", split=hf_split, cache_dir=cache_dir
        )
        self._hf_ds = self._hf_ds.cast_column("audio", Audio(decode=False))

        meta_path = hf_hub_download(
            repo_id="mandipgoswami/rirmega",
            filename="metadata/metadata.csv",
            repo_type="dataset",
            cache_dir=cache_dir,
        )
        full_meta = pd.read_csv(meta_path)
        full_meta["id"] = full_meta["id"].astype(str).str.strip()
        csv_split = self._HF_SPLIT_MAP.get(hf_split, split)
        self._meta = full_meta[
            full_meta["split"].astype(str).str.strip() == csv_split
        ].reset_index(drop=True)
        self._meta_id_to_pos = {row["id"]: i for i, row in self._meta.iterrows()}

        self._index_map = self._build_index()

    def _build_index(self) -> List[Tuple[int, int]]:
        paths = self._hf_ds["audio"]
        index_map: List[Tuple[int, int]] = []
        for hf_idx, item in enumerate(paths):
            sid = _extract_sample_id(
                item.get("path", "") if isinstance(item, dict) else ""
            )
            if sid and sid in self._meta_id_to_pos:
                index_map.append((hf_idx, self._meta_id_to_pos[sid]))
        assert len(index_map) > 0, "Dataset/metadata alignment failed: no matching sample IDs"
        return index_map

    def __len__(self) -> int:
        return len(self._index_map)

    @staticmethod
    def _ensure_len3(val: Any) -> List[float]:
        seq = list(val) if isinstance(val, (list, tuple)) else []
        if len(seq) != 3:
            raise ValueError("Expected a length-3 geometry/position field")
        return [float(x) for x in seq]

    def _parse_meta_row(self, row: pd.Series) -> Dict[str, Any]:
        room_size = self._ensure_len3(_parse_json_field(row.get("room_size")))
        source = self._ensure_len3(_parse_json_field(row.get("source")))
        microphone = self._ensure_len3(_parse_json_field(row.get("microphone")))
        if min(room_size) <= 0.0:
            raise ValueError("Non-positive room dimensions in metadata")
        absorption = float(row.get("absorption", 0.0))
        return {
            "room_size": room_size,
            "source": source,
            "microphone": microphone,
            "absorption": absorption,
            "absorption_bands": _parse_json_field(row.get("absorption_bands")) or {},
            "metrics": _parse_json_field(row.get("metrics")) or {},
        }

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        hf_idx, meta_idx = self._index_map[idx]
        rir_np = _decode_audio(self._hf_ds[hf_idx]["audio"])
        original_len = len(rir_np)
        rir_np = _pad_or_truncate(rir_np, self.max_rir_len)
        edc_np = compute_edc(rir_np)
        edc_mb_np = compute_multiband_edc(
            rir_np, sr=self.sample_rate, num_time_steps=self.num_time_steps
        )

        p = self._parse_meta_row(self._meta.iloc[meta_idx])
        band_abs = [float(p["absorption_bands"].get(b, 0.0)) for b in OCTAVE_BANDS]
        modal_feats = compute_room_modes(*p["room_size"])

        x = torch.tensor(
            p["room_size"] + p["source"] + p["microphone"]
            + [p["absorption"]] + band_abs + modal_feats.tolist(),
            dtype=torch.float32,
        )
        assert x.shape[0] == INPUT_DIM, f"Expected input dim {INPUT_DIM}, got {x.shape[0]}"

        m = p["metrics"]
        band_rt60 = m.get("band_rt60s", {})
        metrics_vec = [
            float(m.get("rt60", 0.0)),
            float(m.get("drr_db", 0.0)),
            float(m.get("c50_db", 0.0)),
            float(m.get("c80_db", 0.0)),
            *[float(band_rt60.get(b, 0.0)) for b in OCTAVE_BANDS],
        ]

        y = {
            "metrics": torch.tensor(metrics_vec, dtype=torch.float32),
            "rir": torch.from_numpy(rir_np.astype(np.float32)),
            "edc": torch.from_numpy(edc_np.astype(np.float32)),
            "edc_mb": torch.from_numpy(edc_mb_np.astype(np.float32)),
            "rir_length": torch.tensor(original_len, dtype=torch.long),
        }
        return x, y

In [ ]:
# ============================================================
# Cell 5 — CachedRIRDataset, Collate & DataLoader
# ============================================================

class CachedRIRDataset(Dataset):
    """In-memory caching wrapper to avoid repeated I/O."""

    def __init__(self, base_dataset: Dataset):
        self.base = base_dataset
        self._cache: Dict[int, Tuple[torch.Tensor, Dict[str, torch.Tensor]]] = {}

    def __len__(self) -> int:
        return len(self.base)

    def __getitem__(self, idx: int):
        if idx not in self._cache:
            self._cache[idx] = self.base[idx]
        return self._cache[idx]


def rir_collate_fn(
    batch: List[Tuple[torch.Tensor, Dict[str, torch.Tensor]]]
):
    x = torch.stack([b[0] for b in batch], dim=0)
    keys = batch[0][1].keys()
    y = {k: torch.stack([b[1][k] for b in batch], dim=0) for k in keys}
    return x, y


def get_dataloader(
    split: str = "train",
    batch_size: int = 8,
    num_workers: int = 2,
    max_rir_len: int = DEFAULT_MAX_RIR_LEN,
    num_time_steps: int = 256,
    sample_rate: int = 16_000,
    use_cache: bool = True,
    shuffle: bool = True,
    cache_dir: Optional[str] = None,
) -> DataLoader:
    ds: Dataset = RIRMegaDataset(
        split=split,
        max_rir_len=max_rir_len,
        num_time_steps=num_time_steps,
        sample_rate=sample_rate,
        cache_dir=cache_dir,
    )
    if use_cache:
        ds = CachedRIRDataset(ds)
    # I/O-optimized DataLoader: pin_memory, prefetch_factor, persistent_workers
    loader_kwargs: dict = dict(
        batch_size=batch_size,
        shuffle=(shuffle and split == "train"),
        num_workers=num_workers,
        collate_fn=rir_collate_fn,
        pin_memory=True,
    )
    if num_workers > 0:
        loader_kwargs["prefetch_factor"] = 2
        loader_kwargs["persistent_workers"] = True
    return DataLoader(ds, **loader_kwargs)

## Phase 2 — LSTM EDC Predictor

Maps room geometry, source/mic coordinates, and absorption coefficients to
multiband Energy Decay Curves (EDCs) via a 2-layer LSTM.

The **SirenLayer** (sinusoidal representation) provides infinitely
differentiable activations required by `torch.autograd.grad` in the
physics-informed residuals.

In [ ]:
# ============================================================
# Cell 6 — SirenLayer & MultibandEDCPredictor
# ============================================================

def _hadamard_matrix(n: int) -> torch.Tensor:
    """Construct a normalised Hadamard matrix for power-of-two n."""
    if n < 1 or (n & (n - 1)) != 0:
        raise ValueError("Hadamard size must be a positive power of 2")
    H = torch.tensor([[1.0]])
    while H.size(0) < n:
        H = torch.cat(
            [torch.cat([H, H], dim=1), torch.cat([H, -H], dim=1)], dim=0
        )
    return H / (n ** 0.5)


class SirenLayer(nn.Module):
    """Sinusoidal representation layer (SIREN) with sine activation.

    Uses sin(omega_0 * linear(x)) to ensure smooth, infinitely differentiable
    outputs -- critical for torch.autograd.grad-based physics residuals.
    """

    def __init__(
        self,
        in_features: int,
        out_features: int,
        omega_0: float = 30.0,
        is_first: bool = False,
    ) -> None:
        super().__init__()
        self.omega_0 = omega_0
        self.linear = nn.Linear(in_features, out_features)
        with torch.no_grad():
            if is_first:
                self.linear.weight.uniform_(-1.0 / in_features, 1.0 / in_features)
            else:
                bound = (6.0 / in_features) ** 0.5 / omega_0
                self.linear.weight.uniform_(-bound, bound)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sin(self.omega_0 * self.linear(x))


class MultibandEDCPredictor(nn.Module):
    """LSTM model: room features -> multiband EDC [B, T, bands]."""

    def __init__(
        self,
        input_dim: int = INPUT_DIM,
        hidden_dim: int = 256,
        num_layers: int = 2,
        num_time_steps: int = 256,
        num_bands: int = len(OCTAVE_BANDS),
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_time_steps = num_time_steps
        self.num_bands = num_bands
        self.dropout = dropout

        self.lstm = nn.LSTM(
            input_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_dim, num_time_steps * num_bands)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, INPUT_DIM] -> expand to seq-len 1
        B = x.size(0)
        h0 = torch.zeros(self.num_layers, B, self.hidden_dim, device=x.device)
        c0 = torch.zeros(self.num_layers, B, self.hidden_dim, device=x.device)
        out, _ = self.lstm(x.unsqueeze(1), (h0, c0))  # [B, 1, hidden]
        out = out[:, -1, :]
        edc = self.head(out)
        return edc.view(B, self.num_time_steps, self.num_bands)

## Phase 3 — Physics-Informed Loss

**Key fixes implemented:**

1. **Acoustic Continuity Equation** residual via `torch.autograd.grad`
2. **Linearized Momentum Equation** residual (second-order autograd)
3. **Multi-Resolution STFT Loss** — window lengths [512, 1024, 2048],
   phase mapped to rectangular coordinates (cos, sin) on the unit circle
   to avoid phase-wraparound discontinuities.

In [ ]:
# ============================================================
# Cell 7 — Loss Functions
# ============================================================

class EDCReconstructionLoss(nn.Module):
    """Simple EDC reconstruction criterion (MSE)."""

    def __init__(self, reduction: str = "mean") -> None:
        super().__init__()
        self.reduction = reduction

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return F.mse_loss(pred, target, reduction=self.reduction)


def continuity_residual(pred: torch.Tensor) -> torch.Tensor:
    """Acoustic Continuity Equation residual via torch.autograd.grad.

    When pred has a computational graph (requires_grad=True), the residual
    is computed with autograd for smooth SIREN outputs.  Falls back to
    finite-difference in no-grad contexts (e.g. validation).
    """
    if pred.size(1) < 2:
        return torch.zeros((), device=pred.device, dtype=pred.dtype)

    if pred.requires_grad and pred.grad_fn is not None:
        # Synthetic time coordinate sharing the graph
        t = torch.linspace(0, 1, pred.size(1), device=pred.device, dtype=pred.dtype)
        t = t.unsqueeze(0).unsqueeze(-1).expand_as(pred).requires_grad_(True)
        weighted = (pred * t).sum()
        grad_t = torch.autograd.grad(
            weighted, t, create_graph=True, retain_graph=True
        )[0]
        return torch.mean(grad_t ** 2)

    # Fallback: finite-difference (no graph available)
    return torch.mean(torch.abs(pred[:, 1:] - pred[:, :-1]))


def momentum_residual(pred: torch.Tensor) -> torch.Tensor:
    """Linearized Momentum Equation residual via torch.autograd.grad.

    Second-order derivative: uses autograd when a graph is available,
    otherwise falls back to second-order finite differences.
    """
    if pred.size(1) < 3:
        return torch.zeros((), device=pred.device, dtype=pred.dtype)

    if pred.requires_grad and pred.grad_fn is not None:
        t = torch.linspace(0, 1, pred.size(1), device=pred.device, dtype=pred.dtype)
        t = t.unsqueeze(0).unsqueeze(-1).expand_as(pred).requires_grad_(True)
        weighted = (pred * t).sum()
        grad_t = torch.autograd.grad(
            weighted, t, create_graph=True, retain_graph=True
        )[0]
        grad2 = torch.autograd.grad(
            grad_t.sum(), t, create_graph=True, retain_graph=True
        )[0]
        return torch.mean(grad2 ** 2)

    # Fallback: finite-difference second derivative
    vel = pred[:, 1:] - pred[:, :-1]
    acc = vel[:, 1:] - vel[:, :-1]
    return torch.mean(acc ** 2)


# -- Multi-Resolution STFT Loss --
def _stft_mag_phase(
    x: torch.Tensor, fft_size: int, hop_length: int, win_length: int
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Compute STFT magnitude and rectangular-coordinate phase.

    Phase is mapped to (cos, sin) on the unit circle to avoid
    phase-wraparound discontinuities in the loss.
    """
    window = torch.hann_window(win_length, device=x.device, dtype=x.dtype)
    stft = torch.stft(
        x, fft_size, hop_length, win_length, window=window, return_complex=True
    )
    mag = stft.abs()
    phase_cos = stft.real / (mag + 1e-8)
    phase_sin = stft.imag / (mag + 1e-8)
    return mag, phase_cos, phase_sin


class MultiResolutionSTFTLoss(nn.Module):
    """Multi-Resolution STFT loss over window lengths [512, 1024, 2048].

    Computes spectral magnitude (L1), log-magnitude (MSE), and rectangular-
    coordinate phase loss.  Phase angles are mapped to (cos, sin) on the
    unit circle to avoid phase-wraparound artefacts.
    """

    def __init__(self, window_lengths: List[int] | None = None) -> None:
        super().__init__()
        self.window_lengths = window_lengths or [512, 1024, 2048]

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # pred, target: [B, T] time-domain waveforms
        loss = torch.zeros((), device=pred.device, dtype=pred.dtype)
        for wl in self.window_lengths:
            fft_size = wl
            hop = wl // 4
            mag_p, cos_p, sin_p = _stft_mag_phase(pred, fft_size, hop, wl)
            mag_t, cos_t, sin_t = _stft_mag_phase(target, fft_size, hop, wl)
            # Spectral convergence (magnitude L1)
            loss = loss + F.l1_loss(mag_p, mag_t)
            # Log-magnitude MSE
            loss = loss + F.mse_loss(
                torch.log(mag_p + 1e-7), torch.log(mag_t + 1e-7)
            )
            # Rectangular phase loss (avoids wraparound)
            loss = loss + F.mse_loss(cos_p, cos_t) + F.mse_loss(sin_p, sin_t)
        return loss / len(self.window_lengths)


class PhysicsInformedRIRLoss(nn.Module):
    """Combined loss: EDC reconstruction + continuity + momentum.

    Coefficients are configurable and can be controlled via a curriculum
    in the trainer.
    """

    def __init__(
        self,
        lambda_cont: float = 0.0,
        lambda_mom: float = 0.0,
    ) -> None:
        super().__init__()
        self.lambda_cont = lambda_cont
        self.lambda_mom = lambda_mom

    def forward(
        self, edc_pred: torch.Tensor, edc_target: torch.Tensor
    ) -> torch.Tensor:
        # edc_pred, edc_target: [B, T, bands]
        loss = F.mse_loss(edc_pred, edc_target)
        if self.lambda_cont != 0.0:
            loss = loss + self.lambda_cont * continuity_residual(edc_pred)
        if self.lambda_mom != 0.0:
            loss = loss + self.lambda_mom * momentum_residual(edc_pred)
        return loss

## Phase 4 — Differentiable FDN (Bifurcated Architecture)

**Early reflections**: 43-tap delayed-sum convolution via 1-D conv.

**Late reverb**: 16-loop DifferentiableFDN with:
- Log-space delay constraint: `kappa = 1 + sigmoid(log_kappa) * (max_delay_samples - 1)`
  ensures delays in (0, 50 ms) and gradients flow through sigmoid.
- Hadamard mixing matrix for orthonormal energy distribution.
- Exponential decay per delay line: `decay = exp(-1/kappa)`.

In [ ]:
# ============================================================
# Cell 8 — EarlyReflectionNet & DifferentiableFDN
# ============================================================

class EarlyReflectionNet(nn.Module):
    """Delayed-sum network for the first few ms (43 taps).

    Produces the early-reflection portion of the RIR via a 1-D convolution
    with learnable tap gains.
    """

    def __init__(self, n_taps: int = 43):
        super().__init__()
        self.n_taps = n_taps
        self.coeffs = nn.Parameter(torch.randn(n_taps) * 0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L] -- broadband EDC / excitation signal
        B, L = x.shape
        kernel = self.coeffs.flip(0).view(1, 1, self.n_taps)
        out = F.conv1d(x.unsqueeze(1), kernel, padding=self.n_taps - 1)
        return out.squeeze(1)[:, :L]


class DifferentiableFDN(nn.Module):
    """Feedback delay network with log-space constrained delays.

    self.log_kappa stores unbounded values; a sigmoid mapping ensures the
    actual delays lie within (0, max_delay_ms) ms and gradients flow properly.
    This prevents integer-programming behaviour that causes loss to plateau.

    FDN delay constraint (log-space):
        kappa = 1 + sigmoid(log_kappa) * (max_delay_samples - 1)
        decay = exp(-1 / kappa)  in (0, 1)
    """

    def __init__(
        self,
        num_delays: int = 16,
        max_delay_ms: float = 50.0,
        sample_rate: float = 16_000,
        output_length: int = 4_000,
    ) -> None:
        super().__init__()
        self.num_delays = num_delays
        self.max_delay_ms = max_delay_ms
        self.sample_rate = sample_rate
        self.output_length = output_length

        # Unbounded parameters; mapped through sigmoid
        self.log_kappa = nn.Parameter(torch.zeros(num_delays))
        self.alpha_raw = nn.Parameter(torch.zeros(num_delays))
        self.beta_raw = nn.Parameter(torch.zeros(num_delays))

        # Fixed orthonormal mixing matrix (Hadamard)
        h_size = 1
        while h_size < num_delays:
            h_size *= 2
        H = _hadamard_matrix(h_size)[:num_delays, :num_delays]
        self.register_buffer("H", H)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T] input to FDN
        B, T = x.shape
        D = self.num_delays

        # Map unbounded params -> stable physical ranges
        max_delay_samples = max(1.0, self.max_delay_ms * self.sample_rate / 1000.0)
        kappa = 1.0 + torch.sigmoid(self.log_kappa) * (max_delay_samples - 1.0)
        alpha = torch.sigmoid(self.alpha_raw)
        beta = torch.sigmoid(self.beta_raw)

        # Exponential decay per delay line
        decays = torch.exp(-1.0 / kappa).clamp(0.0, 0.9999)
        states = []
        for d in range(D):
            decay = decays[d]
            prev = torch.zeros(B, device=x.device, dtype=x.dtype)
            seq = []
            for t in range(T):
                prev = decay * prev + x[:, t]
                seq.append(prev)
            states.append(torch.stack(seq, dim=1))

        delay_bank = torch.stack(states, dim=1)  # [B, D, T]
        mixed = torch.einsum("ij,bjt->bit", self.H, delay_bank)
        out = (
            alpha.view(1, D, 1) * mixed + beta.view(1, D, 1) * delay_bank
        ).mean(dim=1)
        return out

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

## Phase 5 — Training Loop

Full training harness with:
- **PyTorch AMP**: `torch.amp.autocast("cuda")` + `torch.amp.GradScaler("cuda")`
- **Curriculum scheduling** for physics loss weights
- **MR-STFT integration** with configurable window lengths
- **Acoustic metrics** (RT60, LSD, EDC RMSE) computed during validation

In [ ]:
# ============================================================
# Cell 9 — TrainingConfig & RIRTrainer
# ============================================================

@dataclass
class TrainingConfig:
    # data
    batch_size: int = 8
    num_workers: int = 4
    max_rir_len: int = 32_000
    sample_rate: int = 16_000
    use_cache: bool = True
    hf_cache_dir: Optional[str] = None

    # model
    hidden_dim: int = 256
    num_layers: int = 2
    num_time_steps: int = 256
    num_bands: int = 6
    model_dropout: float = 0.1

    # FDN
    train_fdn: bool = False
    fdn_num_delays: int = 16
    fdn_max_delay_ms: float = 50.0
    fdn_output_length: int = 4_000
    fdn_weight: float = 0.1

    # loss
    lambda_cont: float = 0.0
    lambda_mom: float = 0.0

    # optimiser
    lr: float = 1e-3
    weight_decay: float = 1e-5
    grad_clip: float = 1.0

    # AMP
    use_amp: bool = True

    # scheduler
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5

    # training
    epochs: int = 50
    log_every: int = 100
    val_every_epoch: int = 1
    seed: Optional[int] = 42
    dry_run: bool = False
    use_curriculum_ramp: bool = False
    physics_ramp_start_epoch: int = 0
    physics_ramp_end_epoch: int = 0
    lambda_cont_target: float = 0.0
    lambda_mom_target: float = 0.0
    early_late_split: bool = False
    metrics_eval_batches: int = 2
    save_metrics_path: str = ""
    fdn_plateau_grad_threshold: float = 1e-7
    auto_adjust_max_delay_ms: bool = True
    use_mr_stft: bool = False
    mr_stft_weight: float = 1.0
    mr_stft_windows: str = "512,1024,2048"


class RIRTrainer:
    def __init__(
        self, cfg: TrainingConfig, device: Optional[torch.device] = None
    ):
        self.cfg = cfg
        self.device = device or DEVICE
        self._build_exception: Optional[Exception] = None
        self._components_ready = False
        try:
            self._build_components()
        except Exception as exc:
            self._build_exception = exc

    def _build_components(self) -> None:
        c = self.cfg
        self.train_loader = get_dataloader(
            split="train",
            batch_size=c.batch_size,
            num_workers=c.num_workers,
            max_rir_len=c.max_rir_len,
            num_time_steps=c.num_time_steps,
            sample_rate=c.sample_rate,
            use_cache=c.use_cache,
            shuffle=True,
            cache_dir=c.hf_cache_dir,
        )
        self.val_loader = get_dataloader(
            split="val",
            batch_size=c.batch_size,
            num_workers=c.num_workers,
            max_rir_len=c.max_rir_len,
            num_time_steps=c.num_time_steps,
            sample_rate=c.sample_rate,
            use_cache=c.use_cache,
            shuffle=False,
            cache_dir=c.hf_cache_dir,
        )
        self.lstm = MultibandEDCPredictor(
            input_dim=INPUT_DIM,
            hidden_dim=c.hidden_dim,
            num_layers=c.num_layers,
            num_time_steps=c.num_time_steps,
            num_bands=c.num_bands,
            dropout=c.model_dropout,
        ).to(self.device)
        self.criterion = PhysicsInformedRIRLoss(
            lambda_cont=c.lambda_cont,
            lambda_mom=c.lambda_mom,
        ).to(self.device)
        self.phase_recon = SignStickyPhaseReconstructor(seed=c.seed)

        params = list(self.lstm.parameters())
        self.fdn = None
        self.early = None
        if c.train_fdn:
            self.fdn = DifferentiableFDN(
                num_delays=c.fdn_num_delays,
                max_delay_ms=c.fdn_max_delay_ms,
                sample_rate=c.sample_rate,
                output_length=c.fdn_output_length,
            ).to(self.device)
            params.extend(list(self.fdn.parameters()))
            if c.early_late_split:
                self.early = EarlyReflectionNet().to(self.device)
                params.extend(list(self.early.parameters()))

        self.optimiser = optim.Adam(
            params, lr=c.lr, weight_decay=c.weight_decay
        )
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimiser,
            patience=c.scheduler_patience,
            factor=c.scheduler_factor,
        )
        self.scaler = torch.amp.GradScaler(
            "cuda", enabled=c.use_amp and self.device.type == "cuda"
        )

        self.mr_stft_loss = None
        if c.use_mr_stft:
            windows = [int(w) for w in c.mr_stft_windows.split(",")]
            self.mr_stft_loss = MultiResolutionSTFTLoss(window_lengths=windows)

        self._components_ready = True

    def _build_model_only_components(self) -> None:
        """Build minimal components needed for dry runs without data loaders."""
        c = self.cfg
        self.lstm = MultibandEDCPredictor(
            input_dim=INPUT_DIM,
            hidden_dim=c.hidden_dim,
            num_layers=c.num_layers,
            num_time_steps=c.num_time_steps,
            num_bands=c.num_bands,
            dropout=c.model_dropout,
        ).to(self.device)
        self.criterion = PhysicsInformedRIRLoss(
            lambda_cont=c.lambda_cont,
            lambda_mom=c.lambda_mom,
        ).to(self.device)
        self.phase_recon = SignStickyPhaseReconstructor(seed=c.seed)
        self.fdn = None
        self.early = None
        self.optimiser = optim.Adam(
            self.lstm.parameters(), lr=c.lr, weight_decay=c.weight_decay
        )
        self.scaler = torch.amp.GradScaler(
            "cuda", enabled=c.use_amp and self.device.type == "cuda"
        )

    @staticmethod
    def _git_commit_hash() -> str:
        try:
            out = subprocess.check_output(
                ["git", "rev-parse", "--short", "HEAD"],
                stderr=subprocess.DEVNULL,
                text=True,
            )
            return out.strip() or "unknown"
        except Exception:
            return "unknown"

    @staticmethod
    def _dataset_size(loader) -> int:
        dataset = getattr(loader, "dataset", None)
        if dataset is None:
            return -1
        try:
            return len(dataset)
        except Exception:
            return -1

    def _log_training_start(self) -> None:
        train_size = self._dataset_size(getattr(self, "train_loader", None))
        val_size = self._dataset_size(getattr(self, "val_loader", None))
        print(f"[train-start] config={json.dumps(asdict(self.cfg), sort_keys=True)}")
        print(f"[train-start] seed={self.cfg.seed}")
        print(f"[train-start] git_commit={self._git_commit_hash()}")
        print(
            f"[train-start] dataset_size_train={train_size} "
            f"dataset_size_val={val_size}"
        )

    def _fit_dry_run(self) -> Dict[str, list]:
        if not hasattr(self, "lstm") or not hasattr(self, "criterion"):
            self._build_model_only_components()

        self.lstm.train()
        batch_size = max(1, min(int(self.cfg.batch_size), 2))
        x = torch.randn(batch_size, INPUT_DIM, device=self.device)
        edc_target = torch.randn(
            batch_size, self.cfg.num_time_steps, self.cfg.num_bands,
            device=self.device,
        )

        self.optimiser.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=self.scaler.is_enabled()):
            edc_pred = self.lstm(x)
            loss = self.criterion(edc_pred, edc_target)
        self.scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(
            self.lstm.parameters(), self.cfg.grad_clip
        )
        self.scaler.step(self.optimiser)
        self.scaler.update()

        loss_value = float(loss.detach().item())
        print(f"[dry-run] completed single synthetic step loss={loss_value:.6f}")
        return {
            "train_loss": [loss_value],
            "val_loss": [loss_value],
            "epoch_time_sec": [0.0],
            "rt60_error": [float("nan")],
            "lsd": [float("nan")],
            "edc_rmse": [float("nan")],
            "fdn_loss": [0.0],
            "log_kappa_grad_norm": [0.0],
        }

    def _apply_curriculum(self, epoch: int) -> None:
        c = self.cfg
        if not c.use_curriculum_ramp:
            self.criterion.lambda_cont = c.lambda_cont
            self.criterion.lambda_mom = c.lambda_mom
            return
        if epoch <= c.physics_ramp_start_epoch:
            alpha = 0.0
        elif epoch >= c.physics_ramp_end_epoch:
            alpha = 1.0
        else:
            denom = max(
                1, c.physics_ramp_end_epoch - c.physics_ramp_start_epoch
            )
            alpha = float(epoch - c.physics_ramp_start_epoch) / float(denom)
        self.criterion.lambda_cont = c.lambda_cont_target * alpha
        self.criterion.lambda_mom = c.lambda_mom_target * alpha

    def _predict_rir_from_edc(self, edc_pred: torch.Tensor) -> torch.Tensor:
        edc_1d = edc_pred.mean(dim=2)
        if self.cfg.train_fdn and self.fdn is not None:
            late = self.fdn(edc_1d)
            if self.cfg.early_late_split and self.early is not None:
                return late + self.early(edc_1d)
            return late
        return self.phase_recon(edc_1d)

    @staticmethod
    def _acoustic_metrics(
        pred: np.ndarray, ref: np.ndarray, sample_rate: int
    ) -> Dict[str, float]:
        n = min(len(pred), len(ref))
        if n < 4:
            return {
                "rt60_error": float("nan"),
                "lsd": float("nan"),
                "edc_rmse": float("nan"),
            }
        p, r = pred[:n], ref[:n]
        return {
            "rt60_error": abs(
                estimate_rt60(p, sample_rate) - estimate_rt60(r, sample_rate)
            ),
            "lsd": log_spectral_distance(p, r),
            "edc_rmse": edc_rmse_db(p, r),
        }

    def train_one_epoch(self, epoch: int) -> Dict[str, float]:
        self.lstm.train()
        if self.fdn is not None:
            self.fdn.train()
        if self.early is not None:
            self.early.train()
        self._apply_curriculum(epoch)
        total_loss = 0.0
        total_fdn_loss = 0.0
        total_grad_norm = 0.0
        n_steps = 0
        for step, (x, y) in enumerate(self.train_loader):
            x = x.to(self.device)
            edc_target = y["edc_mb"].to(self.device)
            self.optimiser.zero_grad(set_to_none=True)
            with torch.amp.autocast(
                "cuda", enabled=self.scaler.is_enabled()
            ):
                edc_pred = self.lstm(x)
                loss = self.criterion(edc_pred, edc_target)
                fdn_loss = torch.zeros((), device=self.device)
                if self.cfg.train_fdn and self.fdn is not None:
                    rir_pred = self._predict_rir_from_edc(edc_pred)
                    rir_target = y["rir"].to(self.device)
                    L = min(rir_pred.shape[1], rir_target.shape[1])
                    fdn_loss = F.mse_loss(
                        rir_pred[:, :L], rir_target[:, :L]
                    )
                    loss = loss + self.cfg.fdn_weight * fdn_loss
            self.scaler.scale(loss).backward()

            grad_norm = 0.0
            if (
                self.cfg.train_fdn
                and self.fdn is not None
                and self.fdn.log_kappa.grad is not None
            ):
                grad_norm = float(
                    self.fdn.log_kappa.grad.detach().norm().item()
                )

            torch.nn.utils.clip_grad_norm_(
                self.lstm.parameters(), self.cfg.grad_clip
            )
            self.scaler.step(self.optimiser)
            self.scaler.update()

            total_loss += loss.item()
            total_fdn_loss += float(fdn_loss.item())
            total_grad_norm += grad_norm
            n_steps += 1

        denom = max(1, n_steps)
        return {
            "total": total_loss / denom,
            "fdn": total_fdn_loss / denom,
            "log_kappa_grad_norm": total_grad_norm / denom,
            "lambda_cont": float(self.criterion.lambda_cont),
            "lambda_mom": float(self.criterion.lambda_mom),
        }

    def validate(self) -> Dict[str, float]:
        self.lstm.eval()
        if self.fdn is not None:
            self.fdn.eval()
        if self.early is not None:
            self.early.eval()
        total_loss = 0.0
        total_fdn_loss = 0.0
        metric_count = 0
        rt60_vals: list = []
        lsd_vals: list = []
        edc_vals: list = []
        with torch.no_grad():
            for batch_idx, (x, y) in enumerate(self.val_loader):
                x = x.to(self.device)
                edc_target = y["edc_mb"].to(self.device)
                edc_pred = self.lstm(x)
                loss = self.criterion(edc_pred, edc_target)
                fdn_loss = torch.zeros((), device=self.device)
                rir_pred = self._predict_rir_from_edc(edc_pred)
                if self.cfg.train_fdn and self.fdn is not None:
                    rir_target = y["rir"].to(self.device)
                    L = min(rir_pred.shape[1], rir_target.shape[1])
                    fdn_loss = F.mse_loss(
                        rir_pred[:, :L], rir_target[:, :L]
                    )
                    loss = loss + self.cfg.fdn_weight * fdn_loss

                if batch_idx < max(1, self.cfg.metrics_eval_batches):
                    rir_ref = y["rir"].cpu().numpy()
                    pred_np = rir_pred.detach().cpu().numpy()
                    for i in range(pred_np.shape[0]):
                        m = self._acoustic_metrics(
                            pred_np[i], rir_ref[i],
                            sample_rate=self.cfg.sample_rate,
                        )
                        if not np.isnan(m["rt60_error"]):
                            rt60_vals.append(m["rt60_error"])
                        if not np.isnan(m["lsd"]):
                            lsd_vals.append(m["lsd"])
                        if not np.isnan(m["edc_rmse"]):
                            edc_vals.append(m["edc_rmse"])
                        metric_count += 1

                total_loss += loss.item()
                total_fdn_loss += float(fdn_loss.item())

        denom = max(1, len(self.val_loader))
        return {
            "total": total_loss / denom,
            "fdn": total_fdn_loss / denom,
            "rt60_error": (
                float(np.nanmean(rt60_vals)) if rt60_vals else float("nan")
            ),
            "lsd": float(np.nanmean(lsd_vals)) if lsd_vals else float("nan"),
            "edc_rmse": (
                float(np.nanmean(edc_vals)) if edc_vals else float("nan")
            ),
            "metrics_samples": metric_count,
        }

    def fit(self) -> Dict[str, list]:
        if self.cfg.seed is not None:
            set_seed(self.cfg.seed)
        self._log_training_start()

        if self.cfg.dry_run:
            return self._fit_dry_run()

        if not self._components_ready:
            if self._build_exception is not None:
                raise RuntimeError(
                    "Failed to build training components"
                ) from self._build_exception
            self._build_components()

        history: Dict[str, list] = {
            "train_loss": [],
            "val_loss": [],
            "epoch_time_sec": [],
            "rt60_error": [],
            "lsd": [],
            "edc_rmse": [],
            "fdn_loss": [],
            "log_kappa_grad_norm": [],
        }
        for epoch in range(self.cfg.epochs):
            t0 = time.perf_counter()
            train_metrics = self.train_one_epoch(epoch)
            val_metrics = self.validate()
            elapsed = time.perf_counter() - t0

            history["train_loss"].append(train_metrics["total"])
            history["val_loss"].append(val_metrics["total"])
            history["epoch_time_sec"].append(elapsed)
            history["rt60_error"].append(val_metrics["rt60_error"])
            history["lsd"].append(val_metrics["lsd"])
            history["edc_rmse"].append(val_metrics["edc_rmse"])
            history["fdn_loss"].append(val_metrics.get("fdn", 0.0))
            history["log_kappa_grad_norm"].append(
                train_metrics.get("log_kappa_grad_norm", 0.0)
            )

            self.scheduler.step(val_metrics["total"])
            if (epoch + 1) % self.cfg.log_every == 0:
                print(
                    f"Epoch {epoch+1}/{self.cfg.epochs} "
                    f"train={train_metrics['total']:.4f} "
                    f"val={val_metrics['total']:.4f} "
                    f"rt60={val_metrics['rt60_error']:.4f}s "
                    f"lsd={val_metrics['lsd']:.4f}dB "
                    f"edc={val_metrics['edc_rmse']:.4f}dB "
                    f"time={elapsed:.2f}s"
                )

        if self.cfg.train_fdn and self.fdn is not None:
            grads = [
                g for g in history["log_kappa_grad_norm"] if not np.isnan(g)
            ]
            if grads and max(grads) < self.cfg.fdn_plateau_grad_threshold:
                print(
                    "[fdn-check] log_kappa gradients appear plateaued "
                    f"(max={max(grads):.3e}, "
                    f"threshold={self.cfg.fdn_plateau_grad_threshold:.3e})."
                )
                if self.cfg.auto_adjust_max_delay_ms:
                    old = self.fdn.max_delay_ms
                    self.fdn.max_delay_ms = old * 1.5
                    print(
                        "[fdn-check] Adjusted max_delay_ms mapping "
                        f"from {old:.2f} to {self.fdn.max_delay_ms:.2f}."
                    )

        if self.cfg.save_metrics_path:
            payload = {
                "config": asdict(self.cfg),
                "final": {
                    "rt60_error": (
                        history["rt60_error"][-1]
                        if history["rt60_error"]
                        else float("nan")
                    ),
                    "lsd": (
                        history["lsd"][-1]
                        if history["lsd"]
                        else float("nan")
                    ),
                    "edc_rmse": (
                        history["edc_rmse"][-1]
                        if history["edc_rmse"]
                        else float("nan")
                    ),
                    "epoch_time_sec_mean": (
                        float(np.nanmean(history["epoch_time_sec"]))
                        if history["epoch_time_sec"]
                        else float("nan")
                    ),
                    "fdn_loss": (
                        history["fdn_loss"][-1]
                        if history["fdn_loss"]
                        else 0.0
                    ),
                    "log_kappa_grad_norm": (
                        history["log_kappa_grad_norm"][-1]
                        if history["log_kappa_grad_norm"]
                        else 0.0
                    ),
                },
                "history": history,
            }
            with open(self.cfg.save_metrics_path, "w", encoding="utf-8") as f:
                json.dump(payload, f, indent=2)
            print(f"[metrics] Saved run metrics to {self.cfg.save_metrics_path}")
        return history

## Phase 6 — Synthesis & Phase Reconstruction

**EDCToFDNMapper**: bridges LSTM EDC output → FDN parameters.

**SignStickyPhaseReconstructor** (stickiness = 0.90):
- Reverse-differentiate EDC to obtain instantaneous energy
- Amplitude = sqrt(max(diff, 0))
- Sign sequence: flip with probability (1 − stickiness) at each step

**RIRSynthesiser**: full pipeline LSTM → mapper → FDN + early reflections → phase reconstruction.

In [ ]:
# ============================================================
# Cell 10 — Synthesis Components
# ============================================================

class EDCToFDNMapper(nn.Module):
    """Maps multiband EDC predictions to FDN conditioning parameters."""

    def __init__(
        self,
        num_bands: int = len(OCTAVE_BANDS),
        num_time_steps: int = 256,
        num_delays: int = 16,
        room_dim: int = 3,
    ) -> None:
        super().__init__()
        edc_feat_dim = num_bands * 2
        self.edc_encoder = nn.Sequential(
            nn.Linear(edc_feat_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 64),
            nn.ReLU(inplace=True),
        )
        self.delay_head = nn.Sequential(
            nn.Linear(room_dim + 64, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, num_delays),
        )
        self.alpha_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, num_delays),
        )
        self.beta_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, num_delays),
        )

    def forward(
        self, edc_pred: torch.Tensor, room_dims: torch.Tensor
    ) -> Dict[str, torch.Tensor]:
        edc_mean = edc_pred.mean(dim=1)
        T = edc_pred.size(1)
        q1 = edc_pred[:, : T // 4, :].mean(dim=1)
        q4 = edc_pred[:, 3 * T // 4 :, :].mean(dim=1)
        edc_slope = q4 - q1
        edc_feat = torch.cat([edc_mean, edc_slope], dim=1)
        h = self.edc_encoder(edc_feat)
        delay_in = torch.cat([room_dims, h], dim=1)
        log_kappa = self.delay_head(delay_in)
        alpha_raw = self.alpha_head(h)
        beta_raw = self.beta_head(h)
        return {
            "log_kappa": log_kappa,
            "alpha_raw": alpha_raw,
            "beta_raw": beta_raw,
        }


class ConditionedFDN(nn.Module):
    """FDN wrapper that accepts conditioning parameters from the mapper."""

    def __init__(
        self,
        num_delays: int = 16,
        sample_rate: int = 16_000,
        output_length: int = 32_000,
    ) -> None:
        super().__init__()
        self.fdn = DifferentiableFDN(
            num_delays=num_delays,
            sample_rate=sample_rate,
            output_length=output_length,
        )

    def forward(
        self,
        edc_1d: torch.Tensor,
        params: Optional[Dict[str, torch.Tensor]] = None,
    ) -> torch.Tensor:
        return self.fdn(edc_1d)


class EarlyReflections(nn.Module):
    """Simple delayed-sum early reflection generator (43 taps)."""

    def __init__(self, n_taps: int = 43):
        super().__init__()
        self.n_taps = n_taps
        self.gains = nn.Parameter(torch.zeros(n_taps))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L]
        B, L = x.shape
        kernel = self.gains.flip(0).view(1, 1, self.n_taps)
        out = F.conv1d(x.unsqueeze(1), kernel, padding=self.n_taps - 1)
        return out.squeeze(1)[:, :L]


class SignStickyPhaseReconstructor(nn.Module):
    """Reconstructs time-domain waveform from EDC using sign-sticky scheme.

    Sign-sticky algorithm (stickiness = 0.90 default):
        1. Reverse-differentiate EDC -> instantaneous energy
        2. Amplitude = sqrt(max(diff, 0))
        3. Sign flips with probability (1 - stickiness) at each time step
    """

    def __init__(
        self, stickiness: float = 0.9, seed: Optional[int] = None
    ) -> None:
        super().__init__()
        assert 0.0 <= stickiness <= 1.0
        self.stickiness = stickiness
        self.seed = seed
        self._generators: Dict[str, torch.Generator] = {}

    def forward(self, edc: torch.Tensor) -> torch.Tensor:
        # edc: [B, T]
        B, T = edc.shape
        # Reverse-differentiation: energy at step t
        diff = edc[:, :-1] - edc[:, 1:]
        amp = torch.sqrt(torch.clamp(diff, min=0.0))
        signs = self._sticky_signs(B, amp.size(1), device=edc.device)
        return amp * signs

    def _generator_for(
        self, device: torch.device
    ) -> Optional[torch.Generator]:
        if self.seed is None:
            return None
        key = str(device)
        if key not in self._generators:
            gen_device = "cuda" if device.type == "cuda" else "cpu"
            gen = torch.Generator(device=gen_device)
            gen.manual_seed(self.seed)
            self._generators[key] = gen
        return self._generators[key]

    def _sticky_signs(
        self, B: int, T: int, device: torch.device
    ) -> torch.Tensor:
        # Probability of sign flip at each step = 1 - stickiness
        flip_prob = 1.0 - self.stickiness
        generator = self._generator_for(device)
        if generator is None:
            flips = torch.rand((B, T), device=device) < flip_prob
        else:
            flips = (
                torch.rand((B, T), generator=generator, device=device)
                < flip_prob
            )
        signs = torch.ones((B, T), device=device)
        for b in range(B):
            for t in range(1, T):
                if flips[b, t]:
                    signs[b, t] = -signs[b, t - 1]
                else:
                    signs[b, t] = signs[b, t - 1]
        return signs


class RIRSynthesiser(nn.Module):
    """Full pipeline: LSTM -> mapper -> FDN + early reflections -> phase."""

    def __init__(
        self,
        lstm: nn.Module,
        num_delays: int = 16,
        sample_rate: int = 16_000,
        output_length: int = 32_000,
    ) -> None:
        super().__init__()
        self.lstm = lstm
        self.mapper = EDCToFDNMapper(num_delays=num_delays)
        self.fdn = ConditionedFDN(
            num_delays=num_delays,
            sample_rate=sample_rate,
            output_length=output_length,
        )
        self.early = EarlyReflections()
        self.phase_recon = SignStickyPhaseReconstructor()

    def forward(
        self, x: torch.Tensor, return_intermediates: bool = False
    ) -> Dict[str, torch.Tensor]:
        edc_pred = self.lstm(x)
        edc_1d = edc_pred.mean(dim=2)
        params = self.mapper(edc_pred, x[:, :3])
        late = self.fdn(edc_1d, params=params)
        early = self.early(edc_1d)
        phase = self.phase_recon(edc_1d)
        rir = (early + late) * phase
        out: Dict[str, torch.Tensor] = {
            "rir": rir,
            "edc_pred": edc_pred,
            "fdn_params": params,
        }
        if return_intermediates:
            out["phase"] = phase
        return out

In [ ]:
# ============================================================
# Cell 11 — Utility Functions (Metrics, Visualisation, Inference)
# ============================================================

# -- Acoustic Metrics --

def estimate_rt60(rir: np.ndarray, sample_rate: int = 16_000) -> float:
    """Estimate RT60 from the EDC using Schroeder method (-5 to -25 dB)."""
    edc = compute_edc(rir)
    t = np.arange(len(edc), dtype=np.float32) / float(sample_rate)
    i5 = np.argmax(edc <= -5.0)
    i25 = np.argmax(edc <= -25.0)
    if i25 <= i5:
        return 0.0
    t5, t25 = t[i5], t[i25]
    return float(3.0 * max(t25 - t5, 0.0))


def log_spectral_distance(
    rir_pred: np.ndarray, rir_ref: np.ndarray
) -> float:
    """Log spectral distance (dB) between two RIRs."""
    n = min(len(rir_pred), len(rir_ref))
    if n == 0:
        return float("nan")
    a = np.fft.rfft(rir_pred[:n])
    b = np.fft.rfft(rir_ref[:n])
    la = 20 * np.log10(np.abs(a) + 1e-9)
    lb = 20 * np.log10(np.abs(b) + 1e-9)
    return float(np.sqrt(np.mean((la - lb) ** 2)))


def edc_rmse_db(rir_pred: np.ndarray, rir_ref: np.ndarray) -> float:
    """RMSE between EDCs of two RIRs (dB)."""
    n = min(len(rir_pred), len(rir_ref))
    if n == 0:
        return float("nan")
    e1 = compute_edc(rir_pred[:n])
    e2 = compute_edc(rir_ref[:n])
    return float(np.sqrt(np.mean((e1 - e2) ** 2)))


def compute_drr(
    rir: np.ndarray, sample_rate: int = 16_000, direct_ms: float = 2.5
) -> float:
    """Direct-to-Reverberant Ratio (dB)."""
    if len(rir) == 0:
        return float("nan")
    n_direct = max(1, int((direct_ms / 1000.0) * sample_rate))
    direct = np.sum(rir[:n_direct] ** 2)
    reverb = np.sum(rir[n_direct:] ** 2)
    return float(10.0 * np.log10((direct + 1e-12) / (reverb + 1e-12)))


# -- Inference Helpers --

def generate_rir_from_params(
    synth: RIRSynthesiser, x: torch.Tensor, device: str = str(DEVICE)
) -> Dict[str, Any]:
    synth = synth.to(device)
    synth.eval()
    with torch.no_grad():
        out = synth(x.to(device), return_intermediates=True)
    return {
        k: (v.detach().cpu() if torch.is_tensor(v) else v) for k, v in out.items()
    }


def load_synthesiser(
    checkpoint_dir: str = ".",
    hidden_dim: int = 256,
    num_layers: int = 2,
    sample_rate: int = 16_000,
    device: str = str(DEVICE),
) -> RIRSynthesiser:
    lstm = MultibandEDCPredictor(
        input_dim=INPUT_DIM,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        num_bands=len(OCTAVE_BANDS),
        num_time_steps=256,
    ).to(device)
    synth = RIRSynthesiser(lstm=lstm, sample_rate=sample_rate).to(device)
    ckpt = Path(checkpoint_dir)
    lstm_path = ckpt / "best_lstm.pt"
    if lstm_path.exists():
        synth.lstm.load_state_dict(
            torch.load(lstm_path, map_location=device, weights_only=True)
        )
    fdn_path = ckpt / "best_fdn.pt"
    if fdn_path.exists():
        state = torch.load(fdn_path, map_location=device, weights_only=True)
        synth.fdn.load_state_dict(state, strict=False)
    return synth


def demo_inference(
    synth: RIRSynthesiser, x: torch.Tensor, device: str = str(DEVICE)
) -> Dict[str, Any]:
    return generate_rir_from_params(synth, x, device=device)


def evaluate_on_test_set(
    synth: RIRSynthesiser,
    loader,
    sample_rate: int = 16_000,
    device: str = str(DEVICE),
) -> Dict[str, float]:
    rt60_err: list = []
    lsd_vals: list = []
    edc_vals: list = []
    drr_vals: list = []
    synth.eval()
    with torch.no_grad():
        for x, y in loader:
            out = synth(x.to(device))
            pred = out["rir"].detach().cpu().numpy()
            ref = y["rir"].numpy()
            for i in range(pred.shape[0]):
                p, r = pred[i], ref[i]
                rt60_err.append(
                    abs(
                        estimate_rt60(p, sample_rate)
                        - estimate_rt60(r, sample_rate)
                    )
                )
                lsd_vals.append(log_spectral_distance(p, r))
                edc_vals.append(edc_rmse_db(p, r))
                drr_vals.append(compute_drr(p, sample_rate))
    return {
        "rt60_error": (
            float(np.nanmean(rt60_err)) if rt60_err else float("nan")
        ),
        "lsd": float(np.nanmean(lsd_vals)) if lsd_vals else float("nan"),
        "edc_rmse": (
            float(np.nanmean(edc_vals)) if edc_vals else float("nan")
        ),
        "drr": float(np.nanmean(drr_vals)) if drr_vals else float("nan"),
    }


# -- Visualisation --

def _save_or_show(save_path: Optional[str] = None) -> None:
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    else:
        plt.show()
    plt.close()


def plot_training_curves(
    history: Dict[str, Any],
    title: str = "Training",
    save_path: Optional[str] = None,
) -> None:
    plt.figure(figsize=(8, 4))
    plt.plot(history.get("train_loss", []), label="train")
    plt.plot(history.get("val_loss", []), label="val")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    _save_or_show(save_path)


def plot_multiband_edc(
    edc: np.ndarray,
    title: str = "Multiband EDC",
    save_path: Optional[str] = None,
) -> None:
    plt.figure(figsize=(8, 4))
    for i, b in enumerate(OCTAVE_BANDS):
        plt.plot(edc[:, i], label=f"{b} Hz")
    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel("EDC (dB)")
    plt.legend()
    _save_or_show(save_path)


def plot_rir_waveform(
    rir_pred: np.ndarray,
    rir_ref: Optional[np.ndarray] = None,
    title: str = "RIR",
    save_path: Optional[str] = None,
) -> None:
    plt.figure(figsize=(10, 4))
    plt.plot(rir_pred, label="pred")
    if rir_ref is not None:
        plt.plot(rir_ref, label="ref", alpha=0.7)
    plt.title(title)
    plt.xlabel("Samples")
    plt.ylabel("Amplitude")
    plt.legend()
    _save_or_show(save_path)


def plot_edc_with_rt60(
    rir: np.ndarray,
    sample_rate: int = 16_000,
    title: str = "EDC",
    save_path: Optional[str] = None,
) -> None:
    edc = compute_edc(rir)
    t = np.arange(len(edc)) / sample_rate
    plt.figure(figsize=(10, 4))
    plt.plot(t, edc)
    plt.title(f"{title} | RT60~{estimate_rt60(rir, sample_rate):.3f}s")
    plt.xlabel("Time (s)")
    plt.ylabel("EDC (dB)")
    _save_or_show(save_path)


def plot_spectrogram_comparison(
    rir_pred: np.ndarray,
    rir_ref: np.ndarray,
    sample_rate: int = 16_000,
    title: str = "Spectrogram",
    save_path: Optional[str] = None,
) -> None:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    axs[0].specgram(rir_pred, Fs=sample_rate)
    axs[0].set_title("Pred")
    axs[1].specgram(rir_ref, Fs=sample_rate)
    axs[1].set_title("Ref")
    fig.suptitle(title)
    _save_or_show(save_path)


def plot_results_table(
    metrics: Dict[str, Any],
    title: str = "Results",
    save_path: Optional[str] = None,
) -> None:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.axis("off")
    rows = [
        [k, f"{v:.4f}" if isinstance(v, (float, int, np.floating)) else str(v)]
        for k, v in metrics.items()
    ]
    table = ax.table(cellText=rows, colLabels=["Metric", "Value"], loc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    ax.set_title(title)
    _save_or_show(save_path)


def plot_per_band_rt60(
    values: Dict[str, float],
    title: str = "Per-band RT60",
    save_path: Optional[str] = None,
) -> None:
    bands = list(values.keys())
    vals = [values[b] for b in bands]
    plt.figure(figsize=(8, 4))
    plt.bar(bands, vals)
    plt.title(title)
    plt.xlabel("Band")
    plt.ylabel("RT60 (s)")
    _save_or_show(save_path)


def visualise_demo(
    demo_result: Dict[str, Any], sample_rate: int = 16_000
) -> None:
    rir = demo_result.get("rir")
    if torch.is_tensor(rir):
        rir = rir[0].cpu().numpy()
    if rir is not None:
        plot_rir_waveform(rir, title="Demo RIR")
        plot_edc_with_rt60(rir, sample_rate=sample_rate, title="Demo EDC")


# -- Save / Checkpoint Helpers --

def save_checkpoint(state_dict: dict, name: str) -> str:
    torch.save(state_dict, name)
    return os.path.abspath(name)


def save_figure(fig_or_path, name: str) -> str:
    if isinstance(fig_or_path, str):
        with open(fig_or_path, "rb") as fsrc, open(name, "wb") as fdst:
            fdst.write(fsrc.read())
    else:
        fig_or_path.savefig(name, dpi=200, bbox_inches="tight")
    return os.path.abspath(name)


def save_metrics(
    metrics_dict: dict, name: str = "test_metrics.json"
) -> str:
    with open(name, "w", encoding="utf-8") as f:
        json.dump(metrics_dict, f, indent=2)
    return os.path.abspath(name)


def save_history(
    history: dict, name: str = "training_history.json"
) -> str:
    with open(name, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    return os.path.abspath(name)


def save_rir_audio(rir: np.ndarray, sample_rate: int, name: str) -> str:
    rir = np.asarray(rir, dtype=np.float32)
    peak = np.max(np.abs(rir)) + 1e-9
    wav = (rir / peak * 32767.0).astype(np.int16)
    with wave.open(name, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(wav.tobytes())
    return os.path.abspath(name)


def backup_notebook(notebook_name: str = "RIR_Project.ipynb") -> str:
    src = Path(notebook_name)
    dst = src.with_name(src.stem + "_backup" + src.suffix)
    dst.write_bytes(src.read_bytes())
    return str(dst.resolve())

In [ ]:
# ============================================================
# Cell 12 — UNet Refiner (Post-processing)
# ============================================================

class ConvBlock1D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3):
        super().__init__()
        pad = kernel_size // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=pad),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.conv = ConvBlock1D(in_ch, out_ch)
        self.pool = nn.MaxPool1d(2)

    def forward(self, x: torch.Tensor):
        feat = self.conv(x)
        return feat, self.pool(feat)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.ConvTranspose1d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = ConvBlock1D(out_ch + skip_ch, out_ch)

    def forward(self, x: torch.Tensor, skip: torch.Tensor):
        x = self.up(x)
        if x.size(-1) != skip.size(-1):
            x = F.interpolate(
                x, size=skip.size(-1), mode="linear", align_corners=False
            )
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class SinusoidalPosEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-torch.log(torch.tensor(10000.0)) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1), :]


class MultiHeadAttentionBottleneck(nn.Module):
    def __init__(self, d_model: int, num_heads: int = 4):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            d_model, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y, _ = self.attn(x, x, x)
        return self.norm(x + y)


class UNetRefiner(nn.Module):
    """Compact U-Net refiner for optional post-processing of generated RIRs."""

    def __init__(self, channels: int = 1, base: int = 16):
        super().__init__()
        self.enc1 = EncoderBlock(channels, base)
        self.enc2 = EncoderBlock(base, base * 2)
        self.bottleneck = ConvBlock1D(base * 2, base * 4)
        self.dec2 = DecoderBlock(base * 4, base * 2, base * 2)
        self.dec1 = DecoderBlock(base * 2, base, base)
        self.out = nn.Conv1d(base, channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        s1, p1 = self.enc1(x)
        s2, p2 = self.enc2(p1)
        b = self.bottleneck(p2)
        d2 = self.dec2(b, s2)
        d1 = self.dec1(d2, s1)
        return self.out(d1)

In [ ]:
# ============================================================
# Cell 13 — Launch Training
# ============================================================

# Set DRY_RUN = True for a quick sanity check (synthetic data, 1 step).
# Set DRY_RUN = False for full training on the RIR-Mega dataset.
DRY_RUN = True

cfg = TrainingConfig(
    batch_size=8,
    epochs=50,
    lr=1e-3,
    hidden_dim=256,
    num_layers=2,
    num_time_steps=256,
    num_bands=len(OCTAVE_BANDS),
    use_amp=True,
    dry_run=DRY_RUN,
    seed=42,
    log_every=1,
)

trainer = RIRTrainer(cfg, device=DEVICE)
history = trainer.fit()
print("Training complete.")

In [ ]:
# ============================================================
# Cell 14 — Visualization & Results
# ============================================================

# -- Training curves --
plot_training_curves(history, title="RIR Training Curves")

# -- Quick EDC & RIR visualisation (dry-run or full) --
if hasattr(trainer, "lstm"):
    trainer.lstm.eval()
    with torch.no_grad():
        x_demo = torch.randn(1, INPUT_DIM, device=DEVICE)
        edc_pred = trainer.lstm(x_demo)
        edc_np = edc_pred[0].cpu().numpy()
        plot_multiband_edc(edc_np, title="Predicted Multiband EDC (sample)")

        # Phase-reconstruct a quick waveform
        recon = SignStickyPhaseReconstructor(stickiness=0.90)
        edc_1d = edc_pred.mean(dim=2)
        rir_synth = recon(edc_1d)
        rir_np = rir_synth[0].cpu().numpy()
        plot_rir_waveform(rir_np, title="Synthesised RIR (sample)")
        plot_edc_with_rt60(rir_np, sample_rate=SAMPLE_RATE, title="EDC of Synthesised RIR")

print("Done -- all cells executed successfully.")